# 05_04 Dimension Sweep

This sweep asks how the policies scale with latent dimension at a fixed question budget.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "_sweep_helpers.py").exists():
    sys.path.append(str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "notebooks" / "_sweep_helpers.py").exists():
    sys.path.append(str(NOTEBOOK_DIR / "notebooks"))
else:
    sys.path.append(str(Path(r"C:/Users/49160/Adaptive-Onboarding/notebooks")))

from _sweep_helpers import (
    POLICY_STYLE,
    config_label,
    equalize_y_axes,
    load_run,
    load_sweep,
    plot_dimension_spread_lines,
    plot_metric_lines,
    plot_pareto,
    plot_weighted_delta,
    save,
    setup,
    show_sweeps,
    style_ax,
)

setup()

## Select Sweep

In [ ]:
PARAM = "dim"
SWEEP_FILE = None  # paste an exact dim sweep filename here after running one
CONFIG_FILTER = {}  # e.g. {"horizon": 20, "n_items": 50, "n_users": 500}

show_sweeps(PARAM)
sweep = load_sweep(PARAM, sweep_file=SWEEP_FILE, config_filter=CONFIG_FILTER)
print(config_label(sweep))

## Scaling Outcomes

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
plot_metric_lines(
    axes[0], sweep, "dropout_rate",
    y_scale=100,
    xlabel="Latent dimensions",
    ylabel="Episode dropout rate (%)",
    title="Dropout rate",
)
plot_metric_lines(
    axes[1], sweep, "mean_final_d_error",
    xlabel="Latent dimensions",
    ylabel="Mean D-error",
    title="D-error, all episodes",
)
plot_metric_lines(
    axes[2], sweep, "mean_final_d_error_completed",
    xlabel="Latent dimensions",
    ylabel="Mean D-error",
    title="D-error, completed only",
)
equalize_y_axes(axes[1], axes[2])
plot_metric_lines(
    axes[3], sweep, "mean_estimation_error",
    xlabel="Latent dimensions",
    ylabel="Mean estimation error (lower is better)",
    title="Estimation error",
)
fig.suptitle(config_label(sweep), y=1.03, fontsize=9)
plt.tight_layout()
# save(fig, "fig_05_04_dimension_scaling")
plt.show()

## Trait Uncertainty Spread

Because the number of traits changes in this sweep, this view shows the mean marginal variance with a min-to-max band across dimensions.

In [ ]:
metric = "mean_final_d_error_by_dimension"
comparison_policies = [p for p in ["surrogate_unweighted", "surrogate_weighted"] if p in sweep["conditions"][0]["policies"]]

if metric not in next(iter(sweep["conditions"][-1]["policies"].values())):
    print("This sweep was generated before per-dimension D-error was saved. Re-run experiments.sweep to populate it.")
else:
    fig, axes = plt.subplots(1, len(comparison_policies), figsize=(5.8 * len(comparison_policies), 4), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, policy in zip(axes, comparison_policies):
        plot_dimension_spread_lines(
            ax,
            sweep,
            metric,
            policy=policy,
            xlabel="Latent dimensions",
            ylabel="Mean posterior variance" if ax is axes[0] else "",
            title=POLICY_STYLE[policy]["label"],
        )
    fig.suptitle(config_label(sweep), y=1.03, fontsize=9)
    plt.tight_layout()
    # save(fig, "fig_05_04_trait_uncertainty_spread")
    plt.show()

## Exposure and Weighted Delta

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_metric_lines(
    axes[0], sweep, "sensitive_rate",
    y_scale=100,
    xlabel="Latent dimensions",
    ylabel="Sensitive questions / questions asked (%)",
    title="Sensitive exposure rate",
)
plot_weighted_delta(
    axes[1], sweep, "dropout_rate",
    y_scale=100,
    xlabel="Latent dimensions",
    ylabel="Percentage-point delta",
    title="Delta dropout rate",
)
plot_weighted_delta(
    axes[2], sweep, "mean_estimation_error",
    xlabel="Latent dimensions",
    ylabel="Weighted - unweighted",
    title="Delta estimation error",
)
fig.suptitle(config_label(sweep), y=1.03, fontsize=9)
plt.tight_layout()
# save(fig, "fig_05_04_dimension_exposure_delta")
plt.show()

## Notes

- 
